In [2]:
import requests
import pandas as pd

# Use HTTPS and include a User-Agent (Overpass aggressively blocks anonymous scripts)
url = "https://overpass-api.de/api/interpreter"
headers = {
    "User-Agent": "CareAccessProject/1.0 (Data Science Research)"
}

query = """
[out:json][timeout:25];
area["name"="Seattle"]["admin_level"="8"]->.searchArea;
(
  node["amenity"="clinic"](area.searchArea);
  node["amenity"="hospital"](area.searchArea);
  node["healthcare"="clinic"](area.searchArea);
  node["healthcare"="testing_center"](area.searchArea);
);
out body;
"""

print("Querying Overpass API...")
response = requests.get(url, params={'data': query}, headers=headers)

# Check if the request was successful before trying to decode JSON
if response.status_code == 200:
    data = response.json()
    
    clinic_list = []
    for element in data.get('elements', []):
        tags = element.get('tags', {})
        
        if 'name' in tags:
            clinic_list.append({
                'name': tags.get('name'),
                'type': tags.get('amenity', tags.get('healthcare', 'unknown')),
                'address': tags.get('addr:street', ''),
                'city': tags.get('addr:city', 'Seattle'),
                'phone': tags.get('phone', ''),
                'latitude': element.get('lat'),
                'longitude': element.get('lon')
            })

    if clinic_list:
        osm_clinics = pd.DataFrame(clinic_list)
        osm_clinics.to_csv("../data/processed/osm_clinics_geocoded.csv", index=False)
        print(f"Successfully harvested {len(osm_clinics)} clinics!")
        display(osm_clinics.head())
    else:
        print("Query succeeded, but no clinics were found in the response.")
else:
    print(f"Error {response.status_code}: The API did not return JSON.")
    print(f"Server response snippet: {response.text[:500]}")

Querying Overpass API...
Successfully harvested 82 clinics!


,name,type,address,city,phone,latitude,longitude
0,Navos West Seattle Campus,hospital,Southwest Holden Street,Seattle,+1-206-933-7000,47.534445,-122.366224
1,ZoomCare,clinic,North 34th Street,Seattle,,47.649780,-122.350269
2,ZoomCare,clinic,Broadway East,Seattle,,47.623953,-122.321117
3,ZoomCare,clinic,Northwest Market Street,Seattle,+1-844-966-6777,47.668857,-122.383820
4,ZoomCare,clinic,Stone Way North,Seattle,,47.661807,-122.342366


In [3]:
import requests
import pandas as pd

url = "https://overpass-api.de/api/interpreter"
headers = {
    "User-Agent": "CareAccessProject/1.0 (Data Science Research)"
}

# Updated query for Washington State (admin_level 4)
query = """
[out:json][timeout:90];
area["name"="Washington"]["admin_level"="4"]["boundary"="administrative"]->.searchArea;
(
  node["amenity"="hospital"](area.searchArea);
  node["amenity"="clinic"](area.searchArea);
  node["healthcare"="hospital"](area.searchArea);
  node["healthcare"="clinic"](area.searchArea);
);
out body;
"""

print("Querying Overpass API for Washington State...")
response = requests.get(url, params={'data': query}, headers=headers)

if response.status_code == 200:
    data = response.json()
    
    wa_clinic_list = []
    for element in data.get('elements', []):
        tags = element.get('tags', {})
        
        if 'name' in tags:
            wa_clinic_list.append({
                'name': tags.get('name'),
                'type': tags.get('amenity', tags.get('healthcare', 'unknown')),
                'city': tags.get('addr:city', 'Unknown'),
                'latitude': element.get('lat'),
                'longitude': element.get('lon')
            })

    if wa_clinic_list:
        wa_clinics = pd.DataFrame(wa_clinic_list)
        # Save to a new statewide CSV
        wa_clinics.to_csv("../data/processed/wa_clinics_geocoded.csv", index=False)
        print(f"Successfully harvested {len(wa_clinics)} facilities across Washington!")
        display(wa_clinics.head())
    else:
        print("Query succeeded, but no facilities were found.")
else:
    print(f"Error {response.status_code}: {response.text[:500]}")

Querying Overpass API for Washington State...
Successfully harvested 601 facilities across Washington!


,name,type,city,latitude,longitude
0,Navos West Seattle Campus,hospital,Seattle,47.534445,-122.366224
1,Fresenius Kidney Care,clinic,Tacoma,47.193078,-122.409041
2,Poulsbo Medical Clinic,clinic,Poulsbo,47.744215,-122.640650
3,Indigo Urgent Care,clinic,Unknown,47.614590,-122.033726
4,Overlake Clinics - Urgent Care,clinic,Redmond,47.670266,-122.110253
